# Pipelines: Metaflow model training

##  Install dependencies

In [ ]:
#!pip install -r requirements.txt

## Set username

In [ ]:
# Set username for workflows
import os
os.environ["USERNAME"] = "daniel"

## 1. Understand the pipeline

A Metaflow pipeline is a Python class that inherits from `FlowSpec`. Here are the key building blocks:

| Concept | What it does |
|---|---|
| `FlowSpec` | Base class — every pipeline inherits from this |
| `@step` | Marks a method as a pipeline step |
| `Parameter` | A runtime argument passed via CLI (e.g. `--max_depth 5`) |
| `self.next(...)` | Declares the next step — this is how the DAG is defined |
| `self.*` | Any attribute set in a step is automatically persisted as an artifact |
| `@card` | Attaches a visual HTML report to a step, viewable in the browser |

This tutorial's pipeline DAG:

```
start → ingest_data → split_data → train → show_metrics → register_model → end
```

Each step runs in isolation. Metaflow checkpoints all `self.*` artifacts between steps, so any run can be inspected or resumed at any point.

## 2. Implement the flow

In [ ]:
%%writefile metaflow_trainingflow.py
from metaflow import FlowSpec, Parameter, card, current, step
from metaflow.cards import Image, Markdown, Table


class TrainingFlow(FlowSpec):

    max_depth    = Parameter('max_depth',    default=2,   help='Max depth of the random forest')
    n_estimators = Parameter('n_estimators', default=100, help='Number of trees')
    random_state = Parameter('random_state', default=0,   help='Random seed')

    @card
    @step
    def start(self):
        print(f"max_depth={self.max_depth}  n_estimators={self.n_estimators}")
        self.next(self.ingest_data)

    @card
    @step
    def ingest_data(self):
        from sklearn.datasets import load_iris
        iris = load_iris()
        self.X             = iris.data
        self.y             = iris.target
        self.feature_names = list(iris.feature_names)
        self.target_names  = list(iris.target_names)
        print(f"Loaded {len(self.X)} samples, {self.X.shape[1]} features")
        self.next(self.split_data)

    @card
    @step
    def split_data(self):
        from sklearn.model_selection import train_test_split
        # TODO: Split self.X and self.y into self.X_train, self.X_test, self.y_train, self.y_test
        # Use test_size=0.2 and random_state=self.random_state
        self.next(self.train)

    @card
    @step
    def train(self):
        from sklearn.ensemble import RandomForestClassifier
        # TODO: Train a RandomForestClassifier using self.max_depth, self.n_estimators, self.random_state
        # Store the trained model as self.clf
        self.next(self.show_metrics)

    @card(type='blank')
    @step
    def show_metrics(self):
        import matplotlib.pyplot as plt
        from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay

        y_pred        = self.clf.predict(self.X_test)
        self.accuracy = accuracy_score(self.y_test, y_pred)
        report        = classification_report(
            self.y_test, y_pred, target_names=self.target_names, output_dict=True
        )

        # TODO: Add a Markdown title to the card showing the accuracy value
        # TODO: Build rows = [[cls, precision, recall, f1], ...] and add a Table to the card
        # TODO: Create a ConfusionMatrixDisplay figure and add it to the card with Image.from_matplotlib(fig)

        self.next(self.register_model)

    @card
    @step
    def register_model(self):
        import pickle
        # TODO: Save self.clf to 'model.pkl' using pickle
        self.next(self.end)

    @card
    @step
    def end(self):
        print(f"Done. Accuracy: {self.accuracy:.4f}")


if __name__ == '__main__':
    TrainingFlow()

## 3. Run the pipeline

Run it twice with different hyperparameters — we'll compare both runs visually in section 4.

In [ ]:
!python metaflow_trainingflow.py run --max_depth 5 --n_estimators 10 --random_state 0

In [ ]:
# Second run with different hyperparameters — compare both below
!python metaflow_trainingflow.py run --max_depth 2 --n_estimators 50 --random_state 0

## 4. Visualize results

### View step cards

Each step with `@card` generated an HTML report. Open the `show_metrics` card in your browser — it contains the confusion matrix and classification report table:

In [ ]:
# Opens the show_metrics card in your browser
!python metaflow_trainingflow.py card view show_metrics